# Learner Lab: Building a CrewAI Agent — Travel Research

**DIG notebook:** `crewai-travel-research.ipynb`  
**Suggested delivery time:** 35–50 minutes  
**Model:** Amazon Nova Lite through Amazon Bedrock

## Demonstration goal

Build a basic CrewAI solution that researches a destination and produces a practical travel brief. The core demonstration uses one research agent. An extension adds a second itinerary-planning agent to show multi-agent collaboration.

## Learning outcomes

Learners will be able to:

1. Explain the relationship between an **agent**, **tool**, **task**, **crew**, and **process**.
2. Configure a CrewAI `LLM` to use Amazon Bedrock.
3. Give an agent a role, goal, backstory, tools, and behavioural constraints.
4. Define a task with a clear description and expected output.
5. Execute the crew using runtime inputs.
6. Explain where an autonomous agent differs from a simple prompt call.

## Whiteboard before running the notebook

Draw the following flow:

`Traveller request → Crew → Travel Research Agent → Search tool → Research brief`

For the extension:

`Research brief → Itinerary Planner Agent → Day-wise plan`

Ask the class:

- What happens if the agent has no search tool?
- Which decisions should the agent make, and which constraints should the application enforce?
- Why must travel prices, opening hours, visa rules, and safety information be verified close to the travel date?

## Prerequisites and AWS role setup

- Python 3.10–3.12 recommended.
- Bedrock model access in the selected Region.
- Optional: `SERPER_API_KEY` for live Google search through `SerperDevTool`.
- Configure AWS access in the setup cell below. A named AWS profile (including IAM Identity Center/SSO) is recommended; temporary keys can be entered through hidden prompts if your class environment requires them.

### One AWS execution role for this lab

Before class, an AWS administrator should create or assign one role, for example `LearnerBedrockCrewAILabRole`, trusted by the notebook runtime or configured as the target of the learner's AWS profile. Attach a permission policy limited to the model and Region used by the class:

```json
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Action": "bedrock:ListFoundationModels",
      "Resource": "*"
    },
    {
      "Effect": "Allow",
      "Action": "bedrock:InvokeModel",
      "Resource": "arn:aws:bedrock:<REGION>::foundation-model/amazon.nova-lite-v1:0"
    }
  ]
}
```

Replace `<REGION>` with the class Region. If using a cross-Region inference profile, also allow the specific inference-profile ARN and its backing foundation-model ARNs in the Regions used by that profile. The trust policy depends on where the notebook runs, so the administrator must configure it for the actual class environment. Do not grant learners IAM role-creation or administrator permissions.

This is the AWS execution role. The CrewAI researcher and itinerary-planner roles are application-level agents created and used in Steps 6 and the extension; they do not require separate IAM roles. The fallback curated tool works without a search API key. Live travel information must always be verified against authoritative sources before booking.

In [ ]:
%pip install -q -U crewai crewai-tools boto3 litellm nest_asyncio

In [ ]:
# Choose "profile" (recommended) or "keys". Never save credentials in this notebook.
import getpass
import os
import boto3

AUTH_METHOD = "profile"
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")

if AUTH_METHOD == "profile":
    PROFILE_NAME = os.getenv("AWS_PROFILE", "default")
    aws_session = boto3.Session(profile_name=PROFILE_NAME, region_name=AWS_REGION)
    boto3.setup_default_session(profile_name=PROFILE_NAME, region_name=AWS_REGION)
    print(f"Using AWS profile: {PROFILE_NAME}")
elif AUTH_METHOD == "keys":
    print("Use temporary credentials only; input is hidden and is not saved in the notebook.")
    access_key = getpass.getpass("AWS access key ID: ")
    secret_key = getpass.getpass("AWS secret access key: ")
    session_token = getpass.getpass("AWS session token (leave blank if not applicable): ")
    credential_args = {
        "aws_access_key_id": access_key,
        "aws_secret_access_key": secret_key,
        "region_name": AWS_REGION,
    }
    if session_token:
        credential_args["aws_session_token"] = session_token
    aws_session = boto3.Session(**credential_args)
    boto3.setup_default_session(**credential_args)
    del access_key, secret_key, session_token, credential_args
    print("Using temporary credentials entered for this kernel session.")
else:
    raise ValueError('AUTH_METHOD must be either "profile" or "keys".')

print("AWS Region:", AWS_REGION)
print("AWS credentials detected:", aws_session.get_credentials() is not None)

## Step 1 — Imports and configuration

CrewAI uses role-based agents. The `LLM` object connects the agent to the reasoning model. The `tool` decorator turns an ordinary Python function into a capability the agent can select and call.

In [ ]:
import os
import json
from datetime import date

from crewai import Agent, Crew, LLM, Process, Task
from crewai.tools import tool

os.environ["AWS_REGION_NAME"] = AWS_REGION
os.environ["AWS_DEFAULT_REGION"] = AWS_REGION

# Direct Bedrock model ID. If your account requires cross-Region inference,
# use an inference-profile model string such as:
# bedrock/us.amazon.nova-lite-v1:0
BEDROCK_MODEL = os.getenv(
    "BEDROCK_MODEL",
    "bedrock/amazon.nova-lite-v1:0"
)

print("AWS Region:", AWS_REGION)
print("CrewAI model:", BEDROCK_MODEL)
print("Live search enabled:", bool(os.getenv("SERPER_API_KEY")))

## Step 2 — Confirm AWS identity and Bedrock availability

This check uses the AWS identity selected in the setup cell and fails early if credentials are unavailable. It does not print your AWS account ID or caller ARN. Model visibility and invocation rights remain Region- and account-specific.

For a non-default profile, set `AWS_PROFILE` before running the setup cell. For IAM Identity Center, sign in from a terminal first with `aws sso login --profile <profile-name>`. The profile should use or assume the class execution role.

In [ ]:
credentials = aws_session.get_credentials()
if credentials is None:
    raise RuntimeError("AWS credentials were not found. Check the setup cell and AWS profile.")

aws_session.client("sts").get_caller_identity()
bedrock = aws_session.client("bedrock", region_name=AWS_REGION)
models = bedrock.list_foundation_models()["modelSummaries"]
nova_models = [
    (model.get("modelName"), model.get("modelId"))
    for model in models
    if model.get("providerName") == "Amazon" and "Nova" in model.get("modelName", "")
]
print("AWS identity check: passed")
print("Amazon Nova models visible in this Region:")
for name, model_id in nova_models:
    print(f"- {name}: {model_id}")

## Step 3 — Configure Amazon Nova Lite as the CrewAI reasoning model

Temperature is kept low because this agent must organize evidence consistently. For travel marketing copy, a higher value might be acceptable; for facts and constraints, lower variability is preferable.

In [ ]:
travel_llm = LLM(
    model=BEDROCK_MODEL,
    temperature=0.2,
    max_tokens=1800,
    aws_region_name=AWS_REGION,
)

print("LLM configured. No model call has been made yet.")

## Step 4 — Create a predictable fallback research tool

The tool returns curated teaching data for a small set of destinations. It demonstrates tool selection without depending on public internet access.

**Trainer point:** A tool is executable capability, not extra personality text. The LLM decides when and how to call it, but the Python function performs the operation.

In [ ]:
CURATED_TRAVEL_DATA = {
    "kyoto": {
        "best_for": ["temples", "traditional neighbourhoods", "gardens", "seasonal scenery"],
        "areas": ["Gion", "Higashiyama", "Arashiyama", "Central Kyoto"],
        "transport": ["IC transit card", "city bus", "subway", "JR trains for selected routes"],
        "planning_notes": [
            "Start popular sites early to reduce crowding.",
            "Group attractions geographically instead of crossing the city repeatedly.",
            "Some temples and gardens charge separate admission fees.",
            "Opening hours and seasonal access must be checked before travel."
        ],
        "sample_experiences": [
            "Fushimi Inari early-morning walk",
            "Kiyomizu-dera and Higashiyama lanes",
            "Arashiyama bamboo area and nearby temples",
            "Nishiki Market food exploration",
            "Gion evening walk with respectful photography practices"
        ]
    },
    "singapore": {
        "best_for": ["food", "family attractions", "architecture", "efficient transit"],
        "areas": ["Marina Bay", "Chinatown", "Little India", "Sentosa"],
        "transport": ["MRT", "public bus", "walking", "licensed taxi or ride-hailing"],
        "planning_notes": [
            "Allow for hot and humid weather.",
            "Keep indoor alternatives for heavy rain.",
            "Check attraction time slots and closure dates.",
            "Verify current entry requirements before departure."
        ],
        "sample_experiences": [
            "Gardens by the Bay",
            "hawker-centre food tour",
            "Marina Bay evening walk",
            "neighbourhood visit to Chinatown and Little India",
            "Sentosa attractions"
        ]
    }
}

@tool("Search curated destination facts")
def search_curated_destination_facts(query: str) -> str:
    """Search curated classroom travel facts for a destination or planning question."""
    query_lower = query.lower()
    matches = {
        destination: facts
        for destination, facts in CURATED_TRAVEL_DATA.items()
        if destination in query_lower
    }
    if not matches:
        return (
            "No curated destination match. State that live research is required; "
            "do not invent prices, schedules, visa rules, safety advice, or opening hours."
        )
    return json.dumps(matches, indent=2)

print(search_curated_destination_facts.run("Research Kyoto transport and activities"))

## Step 5 — Select tools

When a Serper API key is available, the agent receives both the live-search tool and the curated fallback. Without the key, the demo remains deterministic.

**Security point:** Never paste API keys directly into the notebook. Use environment variables or an approved secrets manager.

In [ ]:
TOOLS = [search_curated_destination_facts]

if os.getenv("SERPER_API_KEY"):
    from crewai_tools import SerperDevTool
    TOOLS.insert(0, SerperDevTool())
    print("Using live Serper search plus curated fallback.")
else:
    print("SERPER_API_KEY not found; using curated fallback only.")

[getattr(t, "name", type(t).__name__) for t in TOOLS]

## Step 6 — Define the travel-research agent

The agent definition answers four questions:

- **Role:** Who is the agent?
- **Goal:** What outcome must it pursue?
- **Backstory:** What expertise and working style should guide it?
- **Tools:** What actions may it perform?

`allow_delegation=False` keeps the basic demo easy to follow. `verbose=True` exposes the agent’s tool-use trace for teaching.

In [ ]:
travel_researcher = Agent(
    role="Evidence-focused Travel Researcher",
    goal=(
        "Research practical, relevant travel options that match the traveller's "
        "destination, duration, interests, pace, and budget. Clearly separate "
        "verified facts, estimates, and items that require confirmation."
    ),
    backstory=(
        "You are an experienced travel researcher. You group activities by location, "
        "avoid unrealistic schedules, and never invent current prices, opening hours, "
        "visa rules, safety information, or availability. When live evidence is absent, "
        "you explicitly tell the traveller what must be verified."
    ),
    llm=travel_llm,
    tools=TOOLS,
    allow_delegation=False,
    verbose=True,
    max_iter=5,
)

print("Agent created:", travel_researcher.role)

## Step 7 — Define the research task

The task description contains runtime placeholders. The `expected_output` acts like an acceptance criterion—it tells the agent what a successful deliverable must contain.

In [ ]:
research_task = Task(
    description="""
Research a trip from {origin} to {destination} for {number_of_days} days.

Traveller profile:
- Interests: {interests}
- Pace: {pace}
- Approximate on-ground budget: {budget}
- Travel period: {travel_period}

Use your available tools. Produce a concise research brief containing:
1. Destination fit and the best areas to focus on
2. Six to eight suitable experiences
3. Practical local-transport guidance
4. A suggested geographic grouping of activities
5. Budget-allocation guidance without inventing live prices
6. Assumptions, uncertainties, and items requiring current verification

Do not present visa, safety, weather, opening-hour, price, or availability claims as
current unless they were obtained from a live source during this run.
""",
    expected_output=(
        "A structured Markdown travel-research brief with headings, concise bullet points, "
        "a shortlist of experiences, planning logic, and a clear verification checklist."
    ),
    agent=travel_researcher,
)

print("Task created.")

## Step 8 — Assemble the crew

Even a one-agent implementation uses a crew as the orchestration container. `Process.sequential` means tasks run in their declared order.

In [ ]:
travel_research_crew = Crew(
    agents=[travel_researcher],
    tasks=[research_task],
    process=Process.sequential,
    verbose=True,
)

print("Crew ready with", len(travel_research_crew.agents), "agent and", len(travel_research_crew.tasks), "task.")

## Step 9 — Run the basic travel-research agent

Change these inputs during class to show that the definitions are reusable. The model and tool calls may incur charges.

In [ ]:
trip_inputs = {
    "origin": "Mumbai",
    "destination": "Kyoto",
    "number_of_days": 5,
    "interests": "culture, vegetarian food, photography, and relaxed walking",
    "pace": "moderate; no more than two major areas per day",
    "budget": "INR 75,000 excluding international flights",
    "travel_period": "November; exact dates not yet selected",
}

research_result = await travel_research_crew.kickoff_async(inputs=trip_inputs)
print("\n===== FINAL RESEARCH BRIEF =====\n")
print(research_result.raw if hasattr(research_result, "raw") else research_result)

## What to point out in the verbose trace

Pause after execution and ask learners to identify:

1. The goal the agent is pursuing.
2. The tool selected and the query passed to it.
3. How the observation returned by the tool changes the final answer.
4. Whether the agent distinguished evidence from assumptions.
5. Whether the final response matches the task’s expected output.

**Critical teaching point:** The agent loop is roughly **reason → select action → call tool → observe result → continue → produce final answer**. CrewAI coordinates this loop; it does not make unreliable source data trustworthy.

# Extension — Add an itinerary-planner agent

The second agent converts research into a day-wise plan. It does not need a search tool because its responsibility is synthesis, not evidence collection.

This extension demonstrates specialization and task handoff.

In [ ]:
itinerary_planner = Agent(
    role="Practical Itinerary Planner",
    goal=(
        "Turn travel research into a realistic day-wise plan that respects geography, "
        "pace, interests, budget guidance, and verification requirements."
    ),
    backstory=(
        "You design achievable itineraries with sensible travel time and recovery time. "
        "You do not silently convert assumptions into facts, and you retain all warnings "
        "about information that requires current verification."
    ),
    llm=travel_llm,
    tools=[],
    allow_delegation=False,
    verbose=True,
)

planning_task = Task(
    description="""
Using the travel research produced by the previous task, create a {number_of_days}-day
itinerary for {destination}. Respect the traveller's interests ({interests}), pace ({pace}),
and approximate budget ({budget}). Group nearby activities and include flexibility.

For every day provide:
- Morning, afternoon, and evening plan
- Area/neighbourhood focus
- Local transport approach
- A lower-energy or bad-weather alternative
- Items that must be checked before the day

Finish with a booking-and-verification checklist. Do not invent live prices or availability.
""",
    expected_output=(
        "A realistic day-wise Markdown itinerary followed by a booking and verification checklist."
    ),
    agent=itinerary_planner,
    context=[research_task],
)

two_agent_crew = Crew(
    agents=[travel_researcher, itinerary_planner],
    tasks=[research_task, planning_task],
    process=Process.sequential,
    verbose=True,
)

print("Two-agent crew ready.")

In [ ]:
# This reruns the research task and then passes its output to the planner.
# It makes additional model/tool calls.
itinerary_result = await two_agent_crew.kickoff_async(inputs=trip_inputs)

print("\n===== FINAL ITINERARY =====\n")
print(itinerary_result.raw if hasattr(itinerary_result, "raw") else itinerary_result)

## Optional — Inspect individual task outputs

Crew output contains the final answer plus task-level outputs. Exact metadata fields can vary by CrewAI version, so inspect the object rather than assuming a fixed UI.

In [ ]:
if hasattr(itinerary_result, "tasks_output"):
    for index, task_output in enumerate(itinerary_result.tasks_output, start=1):
        print(f"\n--- Task {index} output ---")
        print(task_output.raw if hasattr(task_output, "raw") else task_output)
else:
    print("Task-level output metadata is not exposed by this installed CrewAI version.")

## Instructor debrief

### Key concepts to reinforce

| Component | Responsibility in this demo |
|---|---|
| LLM | Reasons and generates decisions/text |
| Agent | Combines role, goal, behavioural guidance, LLM, and tools |
| Tool | Performs an external or deterministic action |
| Task | Defines the assignment and expected output |
| Crew | Orchestrates agents and tasks |
| Process | Controls execution order |
| Context | Passes earlier task output to a later task |

### Knowledge-check questions

1. **Why does the researcher receive tools but the planner does not?**  
   The researcher must acquire evidence; the planner should synthesize the evidence already collected.

2. **What makes this agentic rather than a single prompt?**  
   The agent can reason about the task, choose a tool, invoke it, observe the result, and continue before responding.

3. **Why use `expected_output`?**  
   It defines the deliverable and improves clarity, consistency, and evaluation.

4. **Why use `context=[research_task]`?**  
   It explicitly supplies the research result to the planning task.

5. **Why keep `allow_delegation=False` here?**  
   It makes the execution path controlled and easy to teach. Delegation can be enabled for more autonomous workflows.

### Common pitfalls

- Defining agents with vague roles or overlapping responsibilities.
- Giving every agent every tool.
- Treating a model response as verified travel advice.
- Allowing unrestricted iterations, which increases latency and cost.
- Putting secrets directly in notebook cells.
- Assuming verbose reasoning traces are a substitute for production observability.
- Using open web results without source quality, freshness, safety, and prompt-injection controls.

## Production discussion

Before turning this classroom demo into an application, add:

- Approved and authoritative travel data sources
- Source citations and freshness timestamps
- Input validation and budget/iteration limits
- Guardrails against prompt injection from web content
- Human approval before bookings or purchases
- Tracing, evaluation, latency, token, and cost monitoring
- Privacy controls for traveller information
- Retries and graceful handling of tool/model failures

CrewAI is suited to open-ended collaborative work. If the workflow must follow strict, auditable branches, use a more deterministic flow around the agentic steps.

Official references:

- [CrewAI core concepts](https://docs.crewai.com/core-concepts/Agents)
- [CrewAI tools](https://docs.crewai.com/concepts/tools)
- [CrewAI Serper search tool](https://docs.crewai.com/tools/search-research/serperdevtool)
- [Amazon Nova model information](https://docs.aws.amazon.com/bedrock/latest/userguide/model-cards-amazon.html)

> **Trainer closing:** “CrewAI organizes specialized reasoning and tool use. Production quality still depends on trustworthy data, constrained actions, evaluation, observability, and human control.”